# Máquinas Térmicas – Lección 17
## Modelado termofluídico de un condensador (carcasa y tubos)

**Autor:** Camilo Bayona  
**Fecha:** 20/11/2025


## Objetivos de aprendizaje

- Formular y aplicar **balances de masa** y **energía** para un **condensador shell-&-tube**.
- Identificar regímenes de **flujo interno** (agua en tubos) y **flujo cruzado externo** (vapor en carcasa) mediante **Re**, **Pr**, **Nu**.
- Calcular **coeficientes de convección** (agua/vapor), **conducción** en pared del tubo y **pérdidas** por **convección natural** a ambiente.
- Explicar por qué un condensador/evaporador **garantiza** un cambio de fase, y reconocer que un intercambiador sensible (sin cambio de fase) es un **caso particular** del mismo modelo general.
- Construir el **modelo 0D transitorio** acoplado, generalizado para admitir o no cambio de fase, y describir su implementación numérica.
- Analizar **cámaras de mezcla a presión constante** (mismo principio de los calentadores de agua de alimentación de la Lección 13).
- Identificar qué **instrumentación** (termómetros, manómetros, caudalímetros) se necesita para operar y validar este modelo en una planta real.
- Reconocer que **distintos fluidos de trabajo** (agua, refrigerantes, fluidos orgánicos) usan exactamente las mismas ecuaciones, con sus propias propiedades.

In [ ]:
# Instalación solo en Google Colab; en el entorno local del curso es un no-op
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "matplotlib", "ipywidgets", "numpy", "scipy", "CoolProp"], check=True)

In [ ]:
# === Setup =================================================================
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Circle, Rectangle
from ipywidgets import (interact, interactive_output, FloatSlider, IntSlider,
                        Dropdown, Checkbox, HBox, VBox)
from IPython.display import display
from CoolProp.CoolProp import PropsSI

g0 = 9.81  # m/s^2

# Paleta del proyecto (misma familia que Lecciones 7-16; roles nuevos de esta lección)
COL = {"agua": "#2a78d6", "vapor": "#8a8a86", "pared": "#c3c2b7", "condensacion": "#c0392b",
       "ambiente": "#1b7a3d", "mezcla": "#8e44ad", "perdida": "#e34948", "tinta": "#3a3a38",
       "gas": "#b5651d"}

# --- Geometría asumida (SI) -------------------------------------------------
# El informe original da el NÚMERO de tubos (30, BWG 14) y el paso entre centros
# (ST=18 mm), pero nunca publica D1,D2,L,D_shell numéricamente. Se asume un tubo
# BWG14 de 1/2" (OD estándar de catálogo, espesor BWG14=2.108mm -- consistente con
# que el paso de 18mm sea mayor al OD, como exige la geometría) y una longitud/
# diámetro de carcasa razonables para 30 tubos en arreglo cuadrado alineado.
D2 = 12.70e-3          # m, diámetro EXTERIOR del tubo (1/2" BWG14)
t_wall = 2.108e-3       # m, espesor de pared BWG14 (tabla estándar)
D1 = D2 - 2*t_wall      # m, diámetro INTERIOR del tubo
N_tubes = 30            # Nº de tubos (dato del informe)
ST = 18.0e-3            # m, paso entre centros (dato del informe)
L = 2.5                 # m, longitud efectiva del haz -- asumida
D_shell = 0.30          # m, diámetro interno de la carcasa -- asumida (holgura para 30 tubos a 18mm de paso)
k_tube = 16.0           # W/(m·K), acero inoxidable -- asumido

A_int = N_tubes*math.pi*D1*L          # área interior total (lado agua)
A_ext = N_tubes*math.pi*D2*L          # área exterior total (lado vapor/gas)
A_wall_mean = N_tubes*math.pi*((D1+D2)/2)*L
A_E = math.pi*D_shell*L               # área externa de la carcasa (pérdidas a ambiente)
V_shell_free = math.pi*(D_shell/2)**2*L - N_tubes*math.pi*(D2/2)**2*L   # volumen libre en la carcasa

def props_liquido(fluido, T, p):
    return dict(rho=PropsSI("D","T",T,"P",p,fluido), cp=PropsSI("C","T",T,"P",p,fluido),
                k=PropsSI("L","T",T,"P",p,fluido), mu=PropsSI("V","T",T,"P",p,fluido))

def props_gas(fluido, T, p):
    return props_liquido(fluido, T, p)   # misma llamada CoolProp; el estado (fase) lo decide T,P

print(f"Setup OK -- D1={D1*1e3:.2f} mm  D2={D2*1e3:.2f} mm  N_tubes={N_tubes}  "
      f"A_ext={A_ext:.2f} m²  A_int={A_int:.2f} m²")

 ## 1. Esquema físico y variables

- **Carcasa (shell)**: flujo de vapor compresible, densidad constante aproximada en régimen normal. La carcasa contiene los componentes internos del condensador, incluyendo las placas de soporte de los tubos , las placas tubulares , los tubos , el pozo caliente y las tuberías de extracción [33].

- **Haz de tubos**: Los tubos que pasan por el condensador se ensamblan formando un haz, estos se montan sobre placas tubulares.

- **Tubos**: Flujo de agua refrigerante (fluido incompresible).

- **Geometría**: banco de tubos en arreglo alineado.

- **Áreas características**:
  - **$A_1$**: tubería de ingreso del agua al water box.
  - **$A_2$**: tapa de ingreso de vapor.
  - **$A_3$**: sección de un tubo.
  - **$A_4$**: tubería de salida del condensado.
  - **$A_{int}$**: área interna total de todos los tubos con flujo de agua refrigerante.
  - **$A_{ext}$**: área interna externa de la carcasa donde hay flujo de vapor.
  - **$A_E$**: árewa externa de la carcasa expuesta al ambiente (perdidas).

- **Diámetros / longitudes / pasos**
  - **D_1**: diámetro interior del tubo.
  - **D_2**: diámetro exterior del tubo.
  - **ST**: straight pitch (pitch entre centros, en dirección transversal).
  - **D**: diámetro interno de la carcasa.
  - **L**: longitud efectiva del haz de tubos y carcasa (longitud de intercambio).



In [ ]:
# Diagrama propio del condensador carcasa-tubos (reemplaza las Figuras 1-2 del original,
# consolidando el corte lateral y el detalle de camaras en un solo esquema)
def dibujar_condensador(resaltar=None, n_camaras=4, figsize=(11, 4.6)):
    fig, ax = plt.subplots(figsize=figsize)
    x0, x1, y0, y1 = 1.2, 9.5, 0.3, 3.2
    ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fc="#f4f1ea", ec=COL["tinta"], lw=1.6))

    # haz de tubos: filas de circulos
    n_filas, n_cols = 4, 14
    xs = np.linspace(x0+0.5, x1-0.5, n_cols)
    ys = np.linspace(y0+0.4, y1-0.4, n_filas)
    for yy in ys:
        for xx in xs:
            ax.add_patch(Circle((xx, yy), 0.06, fc="white", ec=COL["agua"], lw=0.8, zorder=3))

    # camaras (zonas) sombreadas sobre el vapor, A_ext
    bordes = np.linspace(x0, x1, n_camaras+1)
    colores_cam = plt.cm.Oranges(np.linspace(0.25, 0.75, n_camaras))
    for k in range(n_camaras):
        ax.axvspan(bordes[k], bordes[k+1], ymin=(y0+0.05)/4.6*1.0, ymax=(y1-0.05)/4.6*1.0,
                   color=colores_cam[k], alpha=0.18, zorder=1)
        ax.text((bordes[k]+bordes[k+1])/2, y1+0.15, f"Cámara {k+1}", ha="center", fontsize=7.5)

    # water box (tapa de ingreso) y tapa de salida
    ax.add_patch(FancyBboxPatch((x0-0.9, y0-0.1), 0.75, y1-y0+0.2, boxstyle="round,pad=0.03",
                                 fc="#dceefb", ec=COL["agua"], lw=1.4))
    ax.text(x0-0.52, (y0+y1)/2, "Water\nbox\n$A_2$", ha="center", va="center", fontsize=7.5)
    ax.add_patch(FancyArrowPatch((x0-1.75, (y0+y1)/2), (x0-0.95, (y0+y1)/2), arrowstyle="-|>",
                                  mutation_scale=13, color=COL["agua"], lw=2))
    ax.text(x0-1.9, (y0+y1)/2+0.25, "agua\n$A_1$", ha="center", fontsize=7.5, color=COL["agua"])
    ax.add_patch(FancyArrowPatch((x1+0.05, (y0+y1)/2), (x1+0.85, (y0+y1)/2), arrowstyle="-|>",
                                  mutation_scale=13, color=COL["agua"], lw=2))
    ax.text(x1+1.0, (y0+y1)/2+0.25, "agua\n(salida)", ha="center", fontsize=7.5, color=COL["agua"])

    # vapor de entrada (arriba) y condensado (abajo, hotwell)
    ax.add_patch(FancyArrowPatch(((x0+x1)/2, y1+0.9), ((x0+x1)/2, y1+0.05), arrowstyle="-|>",
                                  mutation_scale=14, color=COL["vapor"], lw=2.2))
    ax.text((x0+x1)/2, y1+1.05, "vapor (carcasa), $A_{ext}$", ha="center", fontsize=8.5, color=COL["vapor"])
    ax.add_patch(Rectangle((x0+0.3, y0-0.55), x1-x0-0.6, 0.35, fc="#cfe8fb", ec=COL["agua"], lw=1.2))
    ax.text((x0+x1)/2, y0-0.72, "pozo caliente (hotwell)", ha="center", fontsize=7.5)
    ax.add_patch(FancyArrowPatch(((x0+x1)/2, y0-0.6), ((x0+x1)/2, y0-1.05), arrowstyle="-|>",
                                  mutation_scale=12, color=COL["condensacion"], lw=1.8))
    ax.text((x0+x1)/2, y0-1.2, "condensado, $A_4$", ha="center", fontsize=7.5, color=COL["condensacion"])

    ax.text((x0+x1)/2, y1+0.55, "pérdidas al ambiente, $A_E$ (convección natural)", ha="center",
            fontsize=7, color=COL["ambiente"], style="italic")

    ax.set_xlim(x0-2.2, x1+2.1); ax.set_ylim(y0-1.5, y1+1.5); ax.axis("off")
    ax.set_aspect("equal")
    plt.tight_layout(); plt.show()

dibujar_condensador()

## 2. Balance de Masa
La ecuación general del balance de masa en un sistema cerrado  viene dada por
$$
    \frac{d M }{d t} = \sum_i \dot{m}_i \tag{1}
$$
donde $M$ es la masa total del sistema y $\dot{m}_i$ es el i-ésimo flujo (positivo entrante, negativo saliente) del sistema.
Este flujo másico se puede calcular sabiendo la velocidad promedio del flujo $\overline{V}$, la densidad del fluido $\rho$ y el área de sección transversal del canal de flujo $A$, de la forma $\dot{m}=\rho\overline{V}{A}$.

En el caso de un líquido fluyendo en el interior de una tubería, donde no hay acumulación ni compresibilidad del fluido, se tiene que el flujo másico a la entrada es igual a flujo másico de la salida $\dot{m}_1=\dot{m}_2$.
Por el contrario, en el caso del vapor, siendo un gas compresible, se debe evaluar el cambio de masa en el sistema debido a procesos de compresión o expansión.
Sin embargo, debido a que la expansión del vapor dentro de la carcasa no es significativa, dicho proceso de desprecia en el presente trabajo.

### 2.1 Hidráulica del agua refrigerante

Un esquemático de las principales etapas del flujo de agua refrigerante en el condensador se muestra en la Figura 3.

En éste modelo, la identificación de las condiciones del flujo dentro de los tubos es esencial para la cuantificación de la transferencia de calor.
Por lo tanto, se analizan a continuación las condiciones de: (1) entrada del agua refrigerante al equipo, (2) la distribución de agua en el cabezal del banco de tubos, y (3) el interior de cada tubo.
En dicha figura, se presenta también un corte transversal del condensador donde se muestran las áreas transversales de sus elementos: siendo $A_1$ el área transversal de la tubería de ingreso al condensador, $A_2$ el área transversal de la tapa de ingreso al condensador y $A_3$ el área transversal de un solo tubo BGW 14.


Para realizar el análisis del flujo de agua dentro del condensador se parte por determinar la condición de ingreso del agua proveniente de la torre de enfriamiento, la cual es impulsada por una bomba.
Esta condición es un dato conocido, siendo que la planta térmica cuenta con sensores que permiten medir dichas condiciones de entrada al equipo.
Específicamente, se puede obtener el flujo másico de entrada $\dot{m}_{W1}=\rho_{W1}\dot{q}_{W1}$, a partir de la medición del caudal $\dot{q}_{W1}$ y el dato de densidad del agua $\rho_{W1}$.

El widget siguiente resuelve las ecuaciones (2)-(3) (velocidad en el haz de tubos y número de Reynolds del agua) y las análogas (4)-(5) para el vapor, visualizando la cadena **masa → velocidad → régimen de flujo** pedida explícitamente: el mismo caudal másico produce velocidades y regímenes muy distintos según la sección transversal que atraviese.

**Un hallazgo físico real al construir este widget:** con los valores por defecto, la velocidad del vapor en el haz de tubos resulta sorprendentemente alta (decenas de m/s), mucho mayor que la del agua para un caudal másico comparable. La razón no es un error — es que la densidad del vapor a la presión de vacío del condensador (~0.05 kg/m³ a 8 kPa) es **cuatro órdenes de magnitud menor** que la del agua líquida. Por eso los condensadores de vacío reales necesitan carcasas de **gran diámetro** incluso para flujos de vapor relativamente modestos: la masa de vapor equivalente ocupa un volumen enorme a esa densidad.

In [ ]:
# WIDGET 1 -- de masa a velocidad: balance de masa -> velocidad -> Reynolds (ambos lados)
def balance_masa_velocidad(mdot_agua=3.5, mdot_vapor=0.10, T_agua_C=25.0, p_carcasa_kPa=8.0):
    T_agua = T_agua_C + 273.15
    p_carcasa = p_carcasa_kPa*1e3
    pw = props_liquido("Water", T_agua, 101325.0)

    A1 = math.pi*(0.08/2)**2          # tubería de ingreso de agua (asumida, 80mm)
    A3 = math.pi*(D1/2)**2            # seccion de un tubo

    V1 = mdot_agua/(pw["rho"]*A1)
    V3 = mdot_agua/(N_tubes*pw["rho"]*A3)          # ec. (2)
    Re_agua = pw["rho"]*D1*V3/pw["mu"]              # ec. (3)

    T_sat = PropsSI("T","P",p_carcasa,"Q",1,"Water")
    rho_v = PropsSI("D","P",p_carcasa,"Q",1,"Water")
    mu_v = PropsSI("V","P",p_carcasa,"Q",1,"Water")
    A2_vapor = math.pi*(D_shell/2)**2 - N_tubes*math.pi*(D2/2)**2   # seccion libre en la carcasa
    V_S2 = mdot_vapor/(rho_v*A2_vapor)                               # ec. (4)
    Vmax = ST/(ST-D2)*V_S2                                           # ec. (5)
    Re_vapor = rho_v*D2*Vmax/mu_v

    def regimen(Re, limites=(2300, 10000)):
        if Re < limites[0]: return "laminar"
        if Re < limites[1]: return "transicional"
        return "turbulento"

    fig, ax = plt.subplots(figsize=(8.5, 3.2))
    etiquetas = ["agua\n(tubería $A_1$)", "agua\n(tubo $A_3$)", "vapor\n(carcasa $A_2$)", "vapor\n(máx. en haz)"]
    valores = [V1, V3, V_S2, Vmax]
    colores = [COL["agua"], COL["agua"], COL["vapor"], COL["vapor"]]
    ax.bar(etiquetas, valores, color=colores, alpha=0.75)
    for i, v in enumerate(valores):
        ax.text(i, v, f"{v:.2f} m/s", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("velocidad media [m/s]")
    ax.set_title(f"Re agua (tubo) = {Re_agua:.0f} ({regimen(Re_agua)})     "
                 f"Re vapor (haz) = {Re_vapor:.0f} ({regimen(Re_vapor, (100,1000))})")
    plt.tight_layout(); plt.show()

interact(balance_masa_velocidad,
         mdot_agua=FloatSlider(3.5, min=1.0, max=8.0, step=0.5, description="ṁ agua [kg/s]"),
         mdot_vapor=FloatSlider(0.10, min=0.02, max=0.30, step=0.02, description="ṁ vapor [kg/s]"),
         T_agua_C=FloatSlider(25.0, min=10.0, max=40.0, step=1.0, description="T agua ent. [°C]"),
         p_carcasa_kPa=FloatSlider(8.0, min=4.0, max=50.0, step=1.0, description="P carcasa [kPa]"));

Una vez obtenido el caudal de ingreso se procede a realizar los balances másicos con el fin de obtener la velocidad del fluido en la tapa del condensador y en los tubos, donde la sección transversal de la tubería es menor y por consiguiente, varía la velocidad del fluido.
Partiendo de la ecuación de balance de masa, y sabiendo que no hay acumulación de masa dentro de la tubería, se puede determinar el caudal en la sección transversal de la tapa de ingreso al condensador.
Esto es, se sabe que $\dot{m}_{W1}=\dot{m}_{W2}$, siendo $\dot{m}_{W2}$ el flujo másico en la tapa distribuidora. Se puede entonces calcular la velocidad promedio $\overline{V}_{W2}$ con el dato del área transversal de la tapa $A_2$.
El flujo másico en dicha tapa se distribuye en cada uno de los tubos del interior del condensador.
Por lo tanto, $\dot{m}_{W2}= \rho_{W1}\sum\overline{V}_{Wi}{A_i}$, donde $\overline{V}_{Wi}{A_i}$ es el caudal de un iésimo tubo.
Siendo que el condensador cuenta en su interior con 30 tubos BGW 14, se obtiene de la ecuación anterior un valor aproximado de la velocidad del agua que fluye en un solo tubo:
$$
\overline{V}_{W3}=\frac{\dot{m}_{W2}}{30\rho_{W1}{A_3}}. \tag{2}
$$

Obtenida la velocidad del agua al interior de un tubo, se calcula el número de Reynolds del flujo de agua refrigerante.
Este número permite calcular el coeficiente de transferencia de calor por convección al interior de los tubos, e indica el régimen del flujo, ya sea flujo laminar, transicional o turbulento.
En general, el número de Reynolds está definido como la relación
$$
\text{Re}=\frac{\rho{D}\overline{V}}{\mu}, \tag{3}
$$
donde $D$ es una longitud característica (usualmente asociada con el diámetro de la tubería) y $\mu$ es la viscosidad del fluido.
En particular, para el flujo al interior de un tubo, el número de Reynolds viene dado por $\text{Re}=\rho_{W1}{D}_{1}\overline{V}_{W3}/\mu_{W1}$, donde ${D}_{1}$ es el diámetro interno de los tubos BWG y $\mu_{W1}$ es la viscosidad del agua líquida.

Para determinar las condiciones de salida del agua refrigerante se realiza el análisis de los balances de flujos másicos de forma similar al realizado anteriormente para la entrada del agua.
Es fácil observar que el caudal (y la velocidad) en la tapa de salida es el mismo que en la tapa de entrada debido a la conservación de masa.
También, que el caudal (y la velocidad) del agua de refrigeración en la salida del condensador es equivalente al de entrada.


### 2.2 Hidrodinámica del vapor de agua

Para realizar el análisis del flujo del vapor al interior del condensador se parte de los datos sensados por la planta térmica.
En el caso del vapor, se pueden obtener las mediciones del flujo másico de vapor en la turbina, la temperatura y la presión con la cual ingresa el vapor a la turbina, como se muestra en la *Figura 2*.
Estos datos son usados para calcular las condiciones de ingreso del vapor al condensador.


Partiendo de la información medida y suministrada, se realiza el planteamiento de las ecuaciones necesarias para determinar las condiciones del flujo en el banco de tubos.
En el caso del balance de masa en la tubería de vapor, se sabe que $\dot{m}_{S1}=\dot{m}_{S2}$, siendo $\dot{m}_{S1}$ el flujo medido de masa ingresando a la turbina y $\dot{m}_{S2}$ el flujo de masa de vapor que entra al condensador.
Sabiendo el área de sección transversal de la tubería de entrada de vapor al condensador $A_2$, y suponiendo una densidad constante del vapor de agua $\rho_{S2}$, se calcula la velocidad promedio de entrada del vapor como

$$
\overline{V}_{S2}=\frac{\dot{m}_{S1}}{\rho_{S2}A_{2}} \tag{4}
$$

A partir de la ecuación anterior se puede determinar la velocidad del vapor pasando a través del banco de tubos.
El arreglo del banco de tubos es un arreglo cuadrado con los tubos alineados entre sí.
La distancia entre centros de tubos es de 18 mm, tomándose esta distancia como longitud característica.
Denotándose $S_T$ como la distancia radial entre los centros de los tubos y $D_2$ el diámetro exterior de un tubo, la velocidad máxima del vapor a través del banco de tubos se obtiene mediante la expresión
$$
V_{max}=\frac{ST}{ST-D_{2}}{\overline{V}_{S2}} \tag{5}
$$
Con esta velocidad máxima se calcula el número de Reynolds del vapor usando la ecuación (\ref{eq:reynolds}), asumiendo $D_2$ como la longitud característica del flujo.

## 3. Balance de Energía (0D, transitorio)

La ecuación general del balance de energía en un sistema cerrado viene dada por:
$$
 \rho c_p\frac{d T }{d t} \mathcal{V} = \sum_i \dot{m}_i c_p T_i + \sum_j \dot{Q}_j \tag{5}
 $$
donde $T$ es la temperatura del sistema, $\mathcal{V}$ es el volúmen del sistema, $\dot{m}_i c_p T_i$ es la energía transportada por el i-ésimo flujo (positivo entrante, negativo saliente) sobre el sistema y $\dot{Q}_j$ es el j-ésimo flujo de calor sobre el sistema.
La aplicación particular de esta ecuación sobre el agua refrigerante y el vapor se describe a continuación.

### 3.1 Agua refrigerante (en tubos)

Se realiza entonces el balance energético para el agua refrigerante según el esquema mostrado en la *Figura 4*.
El resultado de este análisis es la cuantificación de los cambios de temperatura dentro de los tubos del condensador.
Definiendo $T_W(t)$ la temperatura del agua al interior de los tubos, $T_{W1}$ la temperatura (conocida) de entrada del agua y $T_{W5}$ la temperatura (desconocida) del agua a la salida, $c_{pW}$ el calor específico a presión constante del agua líquida, $\dot{Q}_{1}$ el flujo de calor transferido del vapor al agua y $\mathcal{V}$ el volumen total interior de los tubos, el balance energético del agua resulta en
\begin{align}
\rho \mathcal{V}c_{pW}\frac{dT_{W}}{dt}=30\ \dot{m}_{W3}c_{pW}T_{W1}-\dot{m}_{W5}c_{pW}T_{W5}+\dot{Q}_{1} \tag{6}
\end{align}
En éste trabajo se asume que la temperatura a la salida del agua será la misma que la temperatura dentro del tubo $T_{W5} = T_{W}$.

#### 3.1.1 Números adimensionales
Para determinar el coeficiente de transferencia de calor por convección que influye en la cantidad $\dot{Q}_{1}$ es necesario calcular otros número adimensionales.
En primera medida, se debe calcular el número de Prandtl.
Este es un número adimensional proporcional al cociente entre la difusividad de momento (viscosidad) y la difusividad térmica:
\begin{align}
\text{Pr}=\frac{\nu}{\alpha}=\frac{c_p\mu}{K} \tag{7}
\end{align}
donde $\nu$ es la viscosidad cinemática, $\alpha$ es la difusividad térmica y $K$ es la conductividad térmica del fluido.

También se realiza el cálculo del número de Nusselt, el cual representa el gradiente del perfil adimensional de la temperatura.
Este es un número adimensional utilizado para caracterizar la transferencia de calor por convección.
Se basa en el análisis dimensional y se utiliza por parámetros de similitud para determinar la transferencia de calor de una superficie caliente por la que se convecta el fluido.
Dicho número se define como:
\begin{align}
\text{Nu}=\frac{h}{DK} \tag{8}
\end{align}
donde $h$ es el coeficiente de transferencia de calor por convección, $D$ es una dimensión característica (que se define como el diámetro de la tubería).
El número de Nusselt es la medida de la relación que existe entre la transferencia de calor por convección y la transferencia de calor por conducción.
Las relaciones de transferencia de calor por convección generalmente se expresan escribiendo el número de Nusselt en función del número de Reynolds y del número de Prandtl.
Para el cálculo del número de Nusselt de un fluido que fluye en el interior de un tubo, la literatura recomienda la ecuación de Chilton-Colburn:
\begin{align}
\text{Nu}=0.023\text{Re}^{0.8}\text{Pr}^{0.4} \tag{9}
\end{align}
Entonces una vez obtenidos el número de Reynolds, Prandtl y Nusselt con las expresiones previamente definidas, se procede a utilizar la ecuación (\ref{eq:nusselt}) para determinar el coeficiente de transferencia de calor por convección del agua refrigerante.
Efectivamente, la transferencia de calor por convección en el interior del tubo se puede modelar por medio de la siguiente ecuación:
\begin{align}
\dot{Q}_1= h_{W}(T_{C2}-T_{W})A_{int} \tag{10}
\end{align}
donde $T_{C2}$ es la temperatura de la superficie interior del tubo, $h_{W}$ es el coeficiente de transferencia de calor por convección en el agua interior y $A_{int}$ es el área de transferencia de calor del interior del tubo.
La ecuación de calor por convección  permite determinar la temperatura de la superficie interior del tubo.


### 3.2 Vapor de agua (en carcasa)

Se plantea el balance energético que describe la variación de la temperatura del vapor que fluye en el interior del condensador.
Definiendo $\mathcal{V}_S$ como el volúmen interior de la carcasa, $T_S$ la temperatura del vapor a su paso por los tubos, $T_{S2}$ la temperatura del vapor a la entrada, $T_{S3}$ la temperatura del vapor a la salida, $\rho_{S}$ la densidad del vapor, $c_{pS}$ el calor específico del vapor a presión constante, $\dot{Q}_{1}$ el flujo de calor perdido por el vapor hacia el agua refrigerante, y $\dot{Q}_{2}$ el flujo de calor perdido hacia el medio ambiente a través de la carcasa, el balance de energía en el vapor es el siguiente:

\begin{align}
\rho_S\mathcal{V}_S{c}_{pS}\frac{d\rho T_{S3}(t)}{dt}=\dot{m}_{S2}{c}_{pS}{T}_{S2}-\dot{m}_{S3}{c}_{pS}{T}_{S3}-\dot{Q}_{1}-\dot{Q}_{2} \tag{11}
\end{align}

En éste trabajo se considera que la temperatura de salida del vapor es la misma temperatura a su paso por los tubos, ${T}_{S3}={T}_{S}$.

Para determinar los flujos de calor $\dot{Q}_1$ y $\dot{Q}_{2}$, se emplean las ecuaciones de transferencia de calor por convección.
El procedimiento para calcular dichas cantidades se detalla a continuación.

#### 3.2.1 Números adimensionales

En primera medida, se calcula el flujo de calor transferido por el vapor en su paso a través del banco de tubos. Para ello se plantea la expresión que permite calcular el coeficiente de convección forzada. Usualmente, dicha cantidad se escribe en términos de los números de Reynolds, Prandtl y Nusselt.
El número de Prandtl se calcula usando la ecuación ($7$) con las propiedades del vapor.
La expresión comúnmente usada para calcular el número de Nusselt en términos del Reynolds y Prandtl es la relación de Chilton-Colburn. En el caso particular de vapor fluyendo de manera cruzada a través del banco de tubos, dicha expresión es de la forma:
\begin{align*}
\text{Nu}=C\ (\text{Re})^{m}(\text{Pr})^{n}\left(\frac{\text{Pr}}{\text{Prs}} \right)^{0.25} \tag{12}
\end{align*}
donde que $\text{Prs}$ es el número de Prandtl de película del fluido que se evalua a la temperatura aproximada de la película del fluido sobre la superficie del tubo.
En la literatura se encuentran dispuestos distintos valores para las constantes $C$, $m$ y $n$, las cuales dependen del Reynolds y de la configuración del banco de tubos .
Dichos valores provienen de resultados experimentales.
Para la configuración del arreglo de tubos alineados, se tienen los coeficientes presentados en la siguiente expresión:
\begin{align}
    \text{Nu}(\text{Re},\text{Pr},\text{Prs}) =
     \begin{cases}
       0.9\ \text{Re}^{0.4}\text{Pr}^{0.36}(\text{Pr}/\text{Prs})^{0.25}, &\quad\text{si}\quad\text{Re}\le 100 \\
       0.52\ \text{Re}^{0.5}\text{Pr}^{0.36}(\text{Pr}/\text{Prs})^{0.25}, &\quad\text{si}\quad 100<\text{Re}\le 1000\\
       0.27\ \text{Re}^{0.63}\text{Pr}^{0.36}(\text{Pr}/\text{Prs})^{0.25},  &\quad\text{si}\quad1000<\text{Re}\le 2\times10^5\\
       0.033\ \text{Re}^{0.8}\text{Pr}^{0.8}(\text{Pr}/\text{Prs})^{0.25},  &\quad\text{si}\quad 2\times10^5<\text{Re}\le 2\times10^6
     \end{cases}
     \tag{13}
\end{align}
Dichos valores son específicos para una configuración con un número de tubos menor a $16$ y $0.7<\text{Pr}<500$.

En ese sentido, debido a que el banco de tubos está conformado por menos de 16 filas se debe aplicar el factor de corrección al número Nusselt, $
\text{Nu}|_{NL<16}=F\text{Nu}
$ donde $F$=0.99.
Una vez obtenidos el número de Reynolds, el número de Prandtl y el número de Nusselt corregido, se procede a utilizar la ecuación ($8$) para determinar el coeficiente de transferencia de calor por convección del vapor $h_S$, donde el diámetro externo de los tubos $D_2$ es considerado como la longitud característica.




#### 3.2.2 Convección natural (carcasa–ambiente)
El flujo de calor $\dot{Q}_1$ que se transmite a los tubos, viene dado por la relación
\begin{align}
\dot{Q}_1= h_{S}(T_{S}-T_{C1})A_{ext} \tag{14}
\end{align}
donde $T_C$ es la temperatura en la superficie externa del tubo y $A_{ext}$ es el área de dicha superficie exterior.

En segunda medida, se calcula el flujo de calor perdido por el vapor a la atmósfera en su paso por el condensador.
Se realiza entonces el análisis de la transferencia de calor por convección natural del aire ambiente sobre la superficie de la carcasa del condensador.
Para esto se calculan los números adimensionales de Grashof y Rayleigh que se enuncian a continuación.
El número de Grashof se calcula como
\begin{align}
\text{Gr}=\frac{g\beta(T_S-T_{\infty})L_C^{3}}{\nu^{2}}  \tag{15}
\end{align}
donde $T_{S}$ es la temperatura de la superficie de la carcasa (que igualamos con la del vapor al interior), $T_{\infty}$ es la temperatura ambiente (conocida), $g$ es la aceleración de la gravedad, $L_C$ es una longitud característica (que consideramos como el diámetro de la carcasa) y $\nu$ es la viscosidad cinemática del aire.
El número de Rayleigh se calcula como
\begin{align}
\text{Ra}=\text{Gr}\text{Pr} \tag{16}
\end{align}
Las propiedades del aire ambiente se determinan a la temperatura de película $T_{f}=(T_{S}+T_{\infty})/2$.
Finalmente, se determina el número de Nusselt a partir de la correlación empírica para la convección natural sobre superficies:
\begin{align}
\text{Nu}=\left\{ 0.6+\frac{0.387\
\text{Ra}^{{1}/{6}}}{\lbrack 1+({0.559}/{\text{Pr}})^{{9}/{16}}\rbrack
^{{8}/{27}}}\right\} ^{2}  \tag{17}
\end{align}
Una vez obtenido el número de Nusselt, se calcula el coeficiente de convección natural $h_E$ usando el diámetro de la carcasa $D$ como longitud característica.
Para calcular el calor perdido por convección natural se emplea la relación
\begin{align}
\dot{Q}_{2}= h_EA_{E}(T_{S}-T_{\infty}) \tag{18}
\end{align}
donde el área superficial de la carcasa se calcula como $A_{E}=\pi DL$, siendo $L$ la longitud total de la carcasa.



#### 3.2.3 Intercambio energético a través de la pared del tubo

Planteadas las ecuaciones de transferencia de calor por convección, se hace ahora el análisis de la transferencia de calor por conducción a través de la pared de los tubos:
\begin{align}
\dot{Q}_{1}= KA(T_{C1}-T_{C2}) \tag{19}
\end{align}
donde $K$ es la conductividad térmica del material,  $T_{c(t)}$  es la temperatura en la superficie externa del tubo, $T_{c2(t)}$  hace referencia a la temperatura en la superficie interna del tubo y A es el área de transferencia de calor.
Este calor $\dot{Q}_{1}$ transmitido por conducción en el sólido (tubo) es el mismo transferido por convección desde el vapor o hacia el agua refrigerante.
El área de transferencia de calor por conducción en el tubo es el área exterior equivalente a la sumatoria de las áreas exteriores de todos los tubos que conforman el arreglo del banco de tubos.

El widget siguiente resuelve, para las condiciones que el estudiante elija, **toda la cadena de resistencias térmicas**: convección forzada del agua (ec. 10), convección cruzada del vapor en el banco de tubos (ecs. 12-13), conducción a través de la pared (ec. 19) y convección natural de la carcasa al ambiente (ecs. 14-18) — mostrando cuál resistencia **domina** (suele ser la de menor $h$, casi siempre el lado gas/vapor) y el coeficiente global $U$.

In [ ]:
# WIDGET 2 -- coeficientes de conveccion de toda la cadena de resistencias termicas
def h_tubo_agua(T_agua, mdot_agua):
    pw = props_liquido("Water", T_agua, 101325.0)
    A3 = math.pi*(D1/2)**2
    V3 = mdot_agua/(N_tubes*pw["rho"]*A3)
    Re = pw["rho"]*D1*V3/pw["mu"]; Pr = pw["cp"]*pw["mu"]/pw["k"]
    Nu = 3.66 if Re < 2300 else 0.023*Re**0.8*Pr**0.4
    return Nu*pw["k"]/D1, Re

def h_cruzado_vapor(T_vapor, p_carcasa, mdot_vapor, fluido="Water"):
    rho_v = PropsSI("D","P",p_carcasa,"Q",1,fluido); mu_v = PropsSI("V","P",p_carcasa,"Q",1,fluido)
    k_v = PropsSI("L","P",p_carcasa,"Q",1,fluido); cp_v = PropsSI("C","P",p_carcasa,"Q",1,fluido)
    A2 = math.pi*(D_shell/2)**2 - N_tubes*math.pi*(D2/2)**2
    V_S2 = mdot_vapor/(rho_v*A2); Vmax = ST/(ST-D2)*V_S2
    Re = rho_v*D2*Vmax/mu_v; Pr = cp_v*mu_v/k_v
    if Re <= 100: Cc, m = 0.9, 0.4
    elif Re <= 1000: Cc, m = 0.52, 0.5
    elif Re <= 2e5: Cc, m = 0.27, 0.63
    else: Cc, m = 0.033, 0.8
    Nu = 0.99*Cc*Re**m*Pr**0.36        # factor 0.99: correccion NL<16 filas (eq. texto S3.2.1)
    return Nu*k_v/D2, Re

def h_natural_ambiente(T_vapor, T_amb):
    T_f = (T_vapor+T_amb)/2.0
    air = props_liquido("Air", T_f, 101325.0)
    nu_air = air["mu"]/air["rho"]; beta = 1.0/T_f; Pr = air["cp"]*air["mu"]/air["k"]
    Gr = g0*beta*abs(T_vapor-T_amb)*D_shell**3/nu_air**2
    Ra = Gr*Pr
    Nu = (0.6 + (0.387*Ra**(1/6))/(1+(0.559/Pr)**(9/16))**(8/27))**2
    return Nu*air["k"]/D_shell

def coeficientes_conveccion(mdot_agua=3.5, mdot_vapor=0.10, T_agua_C=25.0, T_vapor_C=40.0,
                             T_amb_C=25.0):
    T_agua, T_vapor, T_amb = T_agua_C+273.15, T_vapor_C+273.15, T_amb_C+273.15
    p_carcasa = PropsSI("P","T",T_vapor,"Q",1,"Water")
    h_w, Re_w = h_tubo_agua(T_agua, mdot_agua)
    h_v, Re_v = h_cruzado_vapor(T_vapor, p_carcasa, mdot_vapor)
    h_E = h_natural_ambiente(T_vapor, T_amb)

    Rs = 1/(h_v*A_ext); Rc = t_wall/(k_tube*A_wall_mean); Rw = 1/(h_w*A_int)
    U = 1/(Rs+Rc+Rw)

    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4))
    nombres = ["$h_W$\n(agua, tubo)", "$h_S$\n(vapor, cruzado)", "$h_E$\n(nat., ambiente)"]
    valores = [h_w, h_v, h_E]
    ax0.bar(nombres, valores, color=[COL["agua"], COL["vapor"], COL["ambiente"]])
    ax0.set_yscale("log"); ax0.set_ylabel("h [W/(m²·K)]  (escala log)")
    for i, v in enumerate(valores):
        ax0.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=9)

    resistencias = [Rs, Rc, Rw]
    nombres_r = ["$R_{vapor}$\n(conv.)", "$R_{pared}$\n(cond.)", "$R_{agua}$\n(conv.)"]
    ax1.bar(nombres_r, resistencias, color=[COL["vapor"], COL["pared"], COL["agua"]])
    ax1.set_ylabel("resistencia térmica [K/W]")
    dominante = nombres_r[int(np.argmax(resistencias))]
    ax1.set_title(f"domina: {dominante.splitlines()[0]}", fontsize=10)
    plt.tight_layout(); plt.show()
    print(f"Re_agua={Re_w:.0f}   Re_vapor={Re_v:.0f}   U_global={U:.2f} W/(m²·K)")

interact(coeficientes_conveccion,
         mdot_agua=FloatSlider(3.5, min=1.0, max=8.0, step=0.5, description="ṁ agua [kg/s]"),
         mdot_vapor=FloatSlider(0.10, min=0.02, max=0.30, step=0.02, description="ṁ vapor [kg/s]"),
         T_agua_C=FloatSlider(25.0, min=10.0, max=40.0, step=1.0, description="T agua [°C]"),
         T_vapor_C=FloatSlider(40.0, min=30.0, max=80.0, step=2.0, description="T vapor [°C]"),
         T_amb_C=FloatSlider(25.0, min=5.0, max=40.0, step=1.0, description="T ambiente [°C]"));

### 3.3 Por qué el condensador *garantiza* un cambio de fase

> **Corrección (2026-11-05).** La §1 del original (más arriba) afirma que "un análisis excluido del modelo es el cambio de fase en el vapor... su inclusión se propone como trabajo futuro." Esto **ya no es así**: el código de la §5 (más abajo) sí implementa la condensación en película (teoría de Nusselt) y calcula la masa condensada en cada sección — el código se extendió después de escribir esa prosa y nunca se actualizó. Esta subsección formaliza esa física, que es justamente la que distingue a un condensador/evaporador de un intercambiador "genérico".

Un intercambiador **sensible** (p. ej. un enfriador de gases, un economizador) transfiere calor cambiando la **temperatura** de ambos fluidos: $\dot Q=\dot m c_p\Delta T$. Un condensador o evaporador transfiere calor cambiando la **fase** de uno de ellos a presión (casi) constante: mientras haya mezcla líquido-vapor presente, la temperatura **no puede cambiar** — está fija en $T_{sat}(P)$ — y todo el calor intercambiado es **calor latente**:

$$\dot Q = \dot m_{cond}\,h_{fg}$$

Esta es la razón física por la que el vapor en la carcasa permanece a $T_S\approx T_{sat}$ durante toda la condensación (supuesto ya usado en la ec. 11 del original) — **no es una simplificación arbitraria, es una consecuencia de la regla de las fases**: con dos fases de una sustancia pura en equilibrio a presión fija, la temperatura queda determinada unívocamente por $P$, sin grados de libertad adicionales.

**El coeficiente de convección también cambia de naturaleza.** La convección de una sola fase (ecs. 9, 12-13) depende del **arrastre del flujo** (Re) — más velocidad, más mezcla, más $h$. La condensación en película, en cambio, depende del **drenaje por gravedad** de la película de condensado que se forma sobre el tubo frío — más película (mayor $\Delta T$ entre vapor y pared), **menor** $h$, porque la película misma es una resistencia térmica adicional. La correlación de Nusselt para condensación en un tubo horizontal es:

$$h_{cond}=0.729\left[\frac{g\,\rho_l(\rho_l-\rho_v)\,k_l^3\,h_{fg}}{\mu_l\,\Delta T\,D_2}\right]^{1/4}$$

con $\rho_l,\mu_l,k_l$ evaluadas a la temperatura de película $T_f=(T_{sat}+T_{pared})/2$, y $\Delta T=T_{sat}-T_{pared}$.

**Generalización pedida explícitamente.** Un intercambiador *sin* cambio de fase no necesita un modelo distinto — es el **mismo modelo general**, con la rama de cambio de fase "apagada": la ec. (11) general ($\rho c_p\,dT/dt=\sum\dot m c_p T-\sum\dot Q$) sigue gobernando la temperatura del fluido caliente, y el coeficiente convectivo vuelve a ser el de una sola fase (ecs. 12-13, no la correlación de Nusselt). El Widget 3 implementa exactamente este interruptor.

In [ ]:
# WIDGET 3 -- meseta de cambio de fase: con cambio de fase (T fija en Tsat, calor
# latente) vs. sin cambio de fase (T evoluciona, calor sensible) -- MISMO modelo general,
# una rama "apagada" o "encendida" segun el checkbox cambio_fase
def meseta_cambio_fase(cambio_fase=True, p_carcasa_kPa=8.0, mdot_vapor=0.10,
                        T_in_sensible_C=150.0, T_out_sensible_C=60.0, mdot_sensible=0.5):
    x = np.linspace(0, 1, 100)   # fraccion acumulada del intercambio de calor

    fig, ax = plt.subplots(figsize=(8, 5.5))
    if cambio_fase:
        p = p_carcasa_kPa*1e3
        T_sat = PropsSI("T","P",p,"Q",1,"Water")
        hfg = PropsSI("H","P",p,"Q",1,"Water") - PropsSI("H","P",p,"Q",0,"Water")
        T_perfil = np.full_like(x, T_sat-273.15)
        m_cond = mdot_vapor   # a estado estacionario, todo el vapor entrante termina condensado
        Q_latente = m_cond*hfg
        ax.plot(x, T_perfil, color=COL["condensacion"], lw=2.6)
        ax.fill_between(x, T_perfil, T_perfil-18, color=COL["condensacion"], alpha=0.12)
        ax.set_title(f"CON cambio de fase: $T$ fija en $T_{{sat}}$={T_sat-273.15:.1f}°C "
                     f"(todo el calor es latente)\n$\\dot Q={Q_latente/1e3:.1f}$ kW   "
                     f"$\\dot m_{{cond}}={m_cond:.3f}$ kg/s   $h_{{fg}}$={hfg/1e3:.0f} kJ/kg")
    else:
        T_in, T_out = T_in_sensible_C+273.15, T_out_sensible_C+273.15
        T_perfil = (T_in + x*(T_out-T_in)) - 273.15
        cp_gas = PropsSI("C","T",(T_in+T_out)/2,"P",101325.0,"Air")
        Q_sensible = mdot_sensible*cp_gas*(T_in-T_out)
        ax.plot(x, T_perfil, color=COL["gas"], lw=2.6)
        ax.fill_between(x, T_perfil, T_perfil.min()-5, color=COL["gas"], alpha=0.12)
        ax.set_title(f"SIN cambio de fase (ej. gas): $T$ cae de {T_in_sensible_C:.0f} a "
                     f"{T_out_sensible_C:.0f}°C (calor sensible)\n$\\dot Q={Q_sensible/1e3:.1f}$ kW   "
                     f"$\\dot m_{{cond}}$ = 0 (no hay cambio de fase)", fontsize=10.5)
    ax.set_xlabel("fracción acumulada del calor intercambiado"); ax.set_ylabel("T [°C]")
    ax.grid(alpha=0.25); ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()

interact(meseta_cambio_fase,
         cambio_fase=Checkbox(True, description="con cambio de fase"),
         p_carcasa_kPa=FloatSlider(8.0, min=4.0, max=50.0, step=1.0, description="P carcasa [kPa]"),
         mdot_vapor=FloatSlider(0.10, min=0.02, max=0.30, step=0.02, description="ṁ vapor [kg/s]"),
         T_in_sensible_C=FloatSlider(150.0, min=60.0, max=300.0, step=10.0, description="T gas ent. [°C]"),
         T_out_sensible_C=FloatSlider(60.0, min=30.0, max=150.0, step=5.0, description="T gas sal. [°C]"),
         mdot_sensible=FloatSlider(0.5, min=0.1, max=2.0, step=0.1, description="ṁ gas [kg/s]"));

## 4. Implementación computacional

En la sección anterior se desarrolló el modelo termo-fluídico del condensador de vapor, en el cual se cuantifica el comportamiento dinámico del agua y el vapor que fluyen por los componentes del condensador de tipo carcasa y tubos.
El modelo completo conduce a un sistema acoplado de varias ecuaciones algebraicas y diferenciales ordinarias ($2$)- ($5$) y ($7$)-($19$).
Además, el modelo matemático dado por esas ecuaciones dinámico (o transitorio) y *0-dimensional*, debido a que gobierna todo el comportamiento temporal y espacial de los fluidos al interior del condensador.
Para completar el modelo, se deben establecer condiciones de compatibilidad entre todos los elementos del análisis: se hace indispensable resolver el sistema acoplado de ecuaciones para obtener un condensador operativo con una respuesta transitoria.


**Incógnitas principales:** $T_W(t)$, $T_S(t)$, $T_{C,1}(t)$, $T_{C,2}(t)$ (si se resuelven explícitas), y campos auxiliares $h_W, h_S, h_E$.  
**Ecuaciones:**  
- Masa → velocidades $\overline V_{W,3}$, $V_{\max}$ → $\mathrm{Re}$.  
- Propiedades → $\mathrm{Pr}$, $\mathrm{Nu}$ → $h$.  
- Compatibilidad de calor → $\dot Q_1$ consistente en (agua, pared, vapor).  
- ODEs de energía → $T_W'(t)$, $T_S'(t)$.

**Condiciones de contorno / iniciales:**  
- $T_W(0)=T_{W,0}$, $T_S(0)=T_{S,0}$.  
- Entradas medidas: $\dot q_{W,1},\,T_{W,1},\,\dot m_{S,1},\,T_{S,2},\,p$.  
- Ambiente: $T_\infty$.  
- Geometría/material: $D_1,D_2,S_T,D,L,K$.  

El diagrama siguiente reemplaza las Figuras 5-8 del original (4 capturas de una herramienta de bloques tipo Simulink, con 13 bloques numerados) por un **solo esquema propio**, agrupando esos 13 bloques en las 4 etapas reales del cálculo: datos de entrada → números adimensionales/convección → balance de energía (las 2 ODEs) → pérdidas ambientales. El contenido es el mismo; el lenguaje visual es el del resto del proyecto.

In [ ]:
# Diagrama de bloques del modelo computacional (consolida las Figuras 5-8 del original)
def dibujar_bloque(titulo, entradas, calculos, salidas, pos=(0,0), figsize=None, ax=None, mostrar=True):
    n = max(len(entradas), len(salidas), 1)
    y_top, y_bot = 2.0, 0.3
    paso = (y_top-y_bot)/max(n-1,1) if n>1 else 0
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=figsize or (9, 3.6))
    ax.add_patch(FancyBboxPatch((pos[0]+2.0, pos[1]+y_bot-0.2), 4.0, (y_top-y_bot)+0.6,
                                 boxstyle="round,pad=0.05", fc="#fdf6e3", ec=COL["tinta"], lw=1.2, ls="--"))
    ax.text(pos[0]+4.0, pos[1]+y_top+0.55, titulo, ha="center", fontsize=9.5, style="italic")
    ax.add_patch(FancyBboxPatch((pos[0]+2.6, pos[1]+(y_top+y_bot)/2-0.45), 2.8, 0.9,
                                 boxstyle="round,pad=0.05", fc="#f4cfcf", ec=COL["tinta"], lw=1.1))
    ax.text(pos[0]+4.0, pos[1]+(y_top+y_bot)/2, "\n".join(calculos), ha="center", va="center", fontsize=7.6)
    for k, txt in enumerate(entradas):
        y = pos[1]+y_top-k*paso
        ax.add_patch(FancyArrowPatch((pos[0]+0.1, y), (pos[0]+2.55, y), arrowstyle="-|>",
                                      mutation_scale=10, color=COL["tinta"]))
        ax.text(pos[0]+0.15, y+0.12, txt, fontsize=7.3)
    for k, txt in enumerate(salidas):
        y = pos[1]+y_top-k*paso
        ax.add_patch(FancyArrowPatch((pos[0]+5.45, y), (pos[0]+7.9, y), arrowstyle="-|>",
                                      mutation_scale=10, color=COL["tinta"]))
        ax.text(pos[0]+6.0, y+0.12, txt, fontsize=7.3)
    if standalone and mostrar:
        ax.set_xlim(pos[0], pos[0]+8); ax.set_ylim(pos[1]+y_bot-0.6, pos[1]+y_top+0.9); ax.axis("off")
        plt.tight_layout(); plt.show()

fig, axs = plt.subplots(2, 2, figsize=(14, 8))
dibujar_bloque("1-2-5. Datos de entrada", ["geometría (D1,D2,ST,D,L)", "agua: ṁ,T,ρ,μ,cp,k",
               "vapor: ṁ,T,p,ρ,μ,cp,k", "ambiente: T∞"], ["agrupa parámetros", "y condiciones", "medidas en planta"],
               ["a bloques 3,6,12"], ax=axs[0,0], mostrar=False)
axs[0,0].set_xlim(0,8); axs[0,0].set_ylim(-0.3,2.9); axs[0,0].axis("off")
dibujar_bloque("3-6-12. Números adimensionales", ["velocidades (masa→vel.)", "propiedades a T,P"],
               ["Re, Pr", "régimen de flujo"], ["a bloques 4,7,8,13"], ax=axs[0,1], mostrar=False)
axs[0,1].set_xlim(0,8); axs[0,1].set_ylim(-0.3,2.9); axs[0,1].axis("off")
dibujar_bloque("4-7-8. Convección y conducción", ["Re, Pr", "¿cambio de fase? (§3.3)"],
               ["Nu → h (forzada o\nNusselt-condensación)", "resistencias R, U global"],
               ["Q₁ a bloques 9,10"], ax=axs[1,0], mostrar=False)
axs[1,0].set_xlim(0,8); axs[1,0].set_ylim(-0.3,2.9); axs[1,0].axis("off")
dibujar_bloque("9-10-11-13. Balance de energía (ODEs)", ["Q₁ (tubo)", "Q₂ (ambiente, conv. natural)"],
               ["dT_W/dt, dT_S/dt", "(o T_S=T_sat si condensa)"], ["T_W(t), T_S(t)", "ṁ_cond(t)"],
               ax=axs[1,1], mostrar=False)
axs[1,1].set_xlim(0,8); axs[1,1].set_ylim(-0.3,2.9); axs[1,1].axis("off")
plt.tight_layout(); plt.show()


En esta sección demostramos la resolución del sistema de ecuaciones mediante el álgebra de bloques en un entorno computacional.
De hecho, la descripción teórica del condensador de vapor no puede realizarse de manera analítica, ya que hay pocas situaciones de operación muy simplificadas en las que se puedan obtener soluciones analíticas transitorias.
La única metodología práctica es aproximar la solución de la operación realista del equipo usando métodos numéricos, y calcular esas aproximaciones en una computadora usando una técnica discreta: por lo general, el *método implícito de cuarto orden Runge-Kutta* se usa para la integración numérica de las derivadas, lo que lleva a un sistema lineal de ecuaciones algebraicas que debe resolverse en cada paso de tiempo.

Un enfoque práctico para resolver este sistema lineal de ecuaciones es implementar la solución numérica de las ecuaciones de gobierno basadas en una resolución explícita de sistema lineal por medio del método de Runge-Kutta de cuarto orden en cada paso de tiempo.
Todo lo anterior, con una interfaz gráfica de usuario para construir _diagramas de bloques algebraícos_ que representan las relaciones entre variables y parámetros dadas por las ecuaciones diferenciales de gobierno.
El enfoque hacia el usuario encapsula todos los tratamientos numéricos dentro de bloques algebraícos, haciendo de la programación computacional un ejercicio ágil y descomplicado que está libre de código.

Por lo tanto, se plantea en el presente trabajo el diagrama de bloques general que representa las relaciones dadas por las ecuaciones de gobierno. Dicho diagrama general se muestra en la Figura 5 , donde el condensador completo está representado en un nivel global.
Esto es, cada sub-bloque del diagrama general en la figura representa un grupo de cálculos para resolver las ecuaciones gobernantes del condensador.
Dichos sub-bloques se detallan conceptualmente en las figuras 6 - 7, donde se presentan los subsistemas que realizan los cálculos.
En cada uno de los subsistemas mostrados en las figuras se enumeran las entradas, los parámetros y las salidas del cálculo.
El diagrama general está conformado por 13 bloques elementales que agrupan en su interior diversos subconjuntos de operaciones en los cuales se resuelven en parte las ecuaciones descritas en el apartado anterior.
Por ejemplo, los sub-bloques calculan: los números adimensionales necesarios para determinar los regímenes de operación de los fluidos, coeficientes de convección y las ecuaciones de balance de masa y energía.



## 5. Codificación en Python

El modelo describe, en régimen transitorio, el proceso de transferencia de calor entre los gases calientes provenientes de la turbina y el agua de enfriamiento que fluye alrededor del tubo.

### Dos modelos del original, fusionados y corregidos

El cuadernillo original traía **dos** implementaciones de código independientes: una (`discretized_condenser_adapted`) más rigurosa —con condensación en película de Nusselt y discretización axial— pero que **no ejecuta tal cual está entregada**, porque invoca un diccionario `default_condenser_params` que nunca se define en ningún lugar del cuadernillo (`NameError` garantizado); y otra (simulación transitoria de la celda final) más simple, sí ejecutable, pero con **propiedades hardcodeadas** (agua a 997 kg/m³, $c_p$=4180 J/kgK fijos, en vez de CoolProp — viola la regla del proyecto de "termodinámica siempre con CoolProp") y un coeficiente de condensación **constante** ($h_{shell}$=5000 W/m²K, sin relación con $\Delta T$).

El modelo siguiente toma la **estructura temporal** que sí funciona de la segunda, la **física de condensación** (Nusselt) de la primera, **propiedades reales vía CoolProp** en ambas ramas, y la **geometría declarada** en el Setup — y lo generaliza con el interruptor `cambio_fase` pedido explícitamente: la misma función resuelve tanto un condensador (rama "encendida") como un intercambiador sensible de gas (rama "apagada"), sin duplicar código.

**Nota de estabilidad numérica.** El lado que condensa evoluciona lentamente (gran capacidad calorífica del agua de enfriamiento, igual que la simulación original de 6 h). El lado sensible, si es un **gas**, tiene una capacidad calorífica diminuta — se verificó numéricamente que su constante de tiempo térmica es del orden de **0.1 s**, miles de veces más rápida que la del agua. Integrar esa rama con el mismo Euler explícito del original diverge (`T` se dispara a valores absurdos). Por eso, cuando `cambio_fase=False`, la temperatura del gas se resuelve **algebraicamente en cada paso** (balance cuasi-estacionario, válido porque su transitorio real es demasiado rápido para animarlo útilmente) — solo $T_W(t)$ (agua) se integra en el tiempo, igual que el original.

In [ ]:
# WIDGET 4 -- simulacion transitoria generalizada (CON/SIN cambio de fase), CoolProp real
def h_tubo_agua_t(T_W, mdot_agua):
    pw = props_liquido("Water", T_W, 101325.0)
    A3 = math.pi*(D1/2)**2
    V3 = mdot_agua/(N_tubes*pw["rho"]*A3)
    Re = pw["rho"]*D1*V3/pw["mu"]; Pr = pw["cp"]*pw["mu"]/pw["k"]
    Nu = 3.66 if Re < 2300 else 0.023*Re**0.8*Pr**0.4
    return Nu*pw["k"]/D1, pw["cp"]

def simular_intercambiador(mdot_agua=3.5, T_agua_in_C=25.0, cambio_fase=True,
                            p_carcasa_kPa=8.0, mdot_vapor=0.10,
                            fluido_sensible="Air", T_gas_in_C=150.0, mdot_gas=0.5,
                            t_final_min=1.0, n_steps=400):
    T_W = T_agua_in_C + 273.15
    t_final = t_final_min*60.0
    dt = t_final/n_steps
    times = np.linspace(0, t_final, n_steps+1)
    TW_arr = np.zeros(n_steps+1); TS_arr = np.zeros(n_steps+1)
    Q_arr = np.zeros(n_steps+1); mcond_arr = np.zeros(n_steps+1)

    if cambio_fase:
        p_shell = p_carcasa_kPa*1e3
        T_sat = PropsSI("T", "P", p_shell, "Q", 1, "Water")
        hfg = PropsSI("H", "P", p_shell, "Q", 1, "Water") - PropsSI("H", "P", p_shell, "Q", 0, "Water")
        TS_arr[0] = T_sat
    else:
        TS_arr[0] = T_gas_in_C + 273.15

    TW_arr[0] = T_W
    m_cond_cum = 0.0
    for i in range(1, n_steps+1):
        h_w, cp_w = h_tubo_agua_t(T_W, mdot_agua)
        if cambio_fase:
            T_wall = min(T_sat - (T_sat-T_W)*0.2, T_sat-0.05)
            for _ in range(40):
                T_wall_c = min(T_wall, T_sat-0.05)   # evita T_film pegado a la curva de saturacion
                T_film = (T_sat+T_wall_c)/2
                liq = props_liquido("Water", T_film, p_shell)
                rho_v = PropsSI("D", "T", T_sat, "Q", 1, "Water")
                dT = max(T_sat-T_wall_c, 0.05)
                h_s = 0.729*((g0*liq["rho"]*(liq["rho"]-rho_v)*liq["k"]**3*hfg) /
                              (liq["mu"]*dT*D2))**0.25
                Rs = 1/(h_s*A_ext); Rc = t_wall/(k_tube*A_wall_mean); Rw = 1/(h_w*A_int)
                T_wall_new = (T_sat*(Rc+Rw) + T_W*Rs)/(Rs+Rc+Rw)
                if abs(T_wall_new-T_wall) < 1e-4:
                    T_wall = T_wall_new; break
                T_wall = T_wall_new
            UA = 1.0/(Rs+Rc+Rw)
            Q_demandado = UA*(T_sat-T_W)
            # la tasa de condensacion no puede superar el vapor realmente disponible --
            # si UA*(Tsat-TW) pide mas calor del que mdot_vapor*hfg puede entregar, el
            # propio Q queda limitado por el suministro de vapor (restriccion fisica real)
            mcond_rate = min(max(Q_demandado,0)/hfg, mdot_vapor)
            Q = mcond_rate*hfg
            m_cond_cum += mcond_rate*dt
            T_S = T_sat
            T_objetivo = T_sat
        else:
            gas = props_liquido(fluido_sensible, TS_arr[i-1], 101325.0)
            A_shell_cross = math.pi*(D_shell/2)**2
            V_shell = mdot_gas/(gas["rho"]*A_shell_cross)
            Vmax = ST/(ST-D2)*V_shell
            Re = gas["rho"]*D2*Vmax/gas["mu"]; Pr = gas["cp"]*gas["mu"]/gas["k"]
            if Re <= 100: Cc, m = 0.9, 0.4
            elif Re <= 1000: Cc, m = 0.52, 0.5
            elif Re <= 2e5: Cc, m = 0.27, 0.63
            else: Cc, m = 0.033, 0.8
            h_s = 0.99*Cc*Re**m*Pr**0.36*gas["k"]/D2
            Rs = 1/(h_s*A_ext); Rc = t_wall/(k_tube*A_wall_mean); Rw = 1/(h_w*A_int)
            UA = 1.0/(Rs+Rc+Rw)
            # balance cuasi-estacionario del lado gas (tau << dt, ver nota de la celda markdown)
            cap_gas = mdot_gas*gas["cp"]
            T_S = (cap_gas*(T_gas_in_C+273.15) + UA*T_W)/(cap_gas+UA)
            Q = UA*(T_S-T_W)
            T_objetivo = T_S

        # Euler IMPLICITO (linealizado sobre UA,T_objetivo del paso) para T_W -- estable para
        # cualquier dt; la relajacion T_W->T_sat (o ->T_S) tiene una constante de tiempo de
        # pocos segundos para esta geometria, demasiado rapida para el Euler explicito original.
        a_relax = UA/(mdot_agua*cp_w)
        T_W = (T_W + dt*a_relax*T_objetivo)/(1+dt*a_relax)
        TW_arr[i] = T_W; TS_arr[i] = T_S; Q_arr[i] = Q; mcond_arr[i] = m_cond_cum

    return times, TW_arr, TS_arr, Q_arr, mcond_arr

def widget_transitorio(mdot_agua=3.5, T_agua_in_C=25.0, cambio_fase=True,
                        p_carcasa_kPa=8.0, mdot_vapor=0.10,
                        fluido_sensible="Air", T_gas_in_C=150.0, mdot_gas=0.5,
                        t_final_min=1.0):
    t, TW, TS, Q, mc = simular_intercambiador(mdot_agua, T_agua_in_C, cambio_fase,
                                               p_carcasa_kPa, mdot_vapor,
                                               fluido_sensible, T_gas_in_C, mdot_gas, t_final_min)
    t_min = t/60.0
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4.6))
    ax0.plot(t_min, TW-273.15, color=COL["agua"], lw=2.2, label="T agua (tubo)")
    ax0.plot(t_min, TS-273.15, color=(COL["condensacion"] if cambio_fase else COL["gas"]), lw=2.2,
             ls="--", label=f"T {'vapor (Tsat)' if cambio_fase else 'gas'} (carcasa)")
    ax0.set_xlabel("tiempo [min]"); ax0.set_ylabel("T [°C]"); ax0.legend(fontsize=9)
    ax0.grid(alpha=0.25); ax0.spines[["top","right"]].set_visible(False)

    ax1.plot(t_min, Q/1e3, color=COL["perdida"], lw=2.0)
    ax1.set_xlabel("tiempo [min]"); ax1.set_ylabel("$\\dot Q$ [kW]")
    ax1.grid(alpha=0.25); ax1.spines[["top","right"]].set_visible(False)
    titulo = (f"condensado acumulado = {mc[-1]:.2f} kg" if cambio_fase
              else f"ṁ_cond = 0 (sin cambio de fase, fluido={fluido_sensible})")
    ax1.set_title(titulo, fontsize=10)
    plt.tight_layout(); plt.show()

interact(widget_transitorio,
         mdot_agua=FloatSlider(3.5, min=1.0, max=8.0, step=0.5, description="ṁ agua [kg/s]"),
         T_agua_in_C=FloatSlider(25.0, min=10.0, max=35.0, step=1.0, description="T agua ent. [°C]"),
         cambio_fase=Checkbox(True, description="con cambio de fase"),
         p_carcasa_kPa=FloatSlider(8.0, min=4.0, max=50.0, step=1.0, description="P carcasa [kPa]"),
         mdot_vapor=FloatSlider(0.10, min=0.02, max=0.30, step=0.02, description="ṁ vapor [kg/s]"),
         fluido_sensible=Dropdown(options=["Air", "Nitrogen"], value="Air", description="fluido (sin cambio fase)"),
         T_gas_in_C=FloatSlider(150.0, min=60.0, max=300.0, step=10.0, description="T gas ent. [°C]"),
         mdot_gas=FloatSlider(0.5, min=0.1, max=2.0, step=0.1, description="ṁ gas [kg/s]"),
         t_final_min=FloatSlider(1.0, min=0.25, max=10.0, step=0.25, description="t final [min]"));

## 6. Cámaras de mezcla a presión constante

No todo intercambio de energía entre corrientes de vapor/agua ocurre a través de una pared (convección + conducción, como en el condensador). En una **cámara de mezcla por contacto directo** (el mismo principio del calentador de agua de alimentación abierto de la Lección 13), dos corrientes a la **misma presión** se combinan físicamente — **se conserva la masa y la energía, no hay pared ni resistencia térmica**:

$$\dot m_3 = \dot m_1+\dot m_2, \qquad \dot m_3 h_3 = \dot m_1 h_1 + \dot m_2 h_2 \qquad (P_1=P_2=P_3)$$

El resultado $h_3$ puede corresponder a una **calidad** $x_3$ intermedia (si $h_3$ cae dentro del domo), incluso si ninguna de las dos corrientes de entrada estaba exactamente ahí — es la manera formal de responder a "mezcladores y cámaras que conservan la presión pero intercambian estados y calidades del vapor." Es, en esencia, la misma ecuación de balance de energía del condensador (ec. 11), pero sin el término de transferencia de calor $\dot Q$ entre dos sustancias separadas por una pared — aquí las dos corrientes **son** la misma sustancia, mezclándose directamente.

In [ ]:
# WIDGET 5 -- camara de mezcla a presion constante: dos corrientes, misma P, distinta calidad
def camara_mezcla(P_kPa=200.0, m1=2.0, x1=0.0, m2=0.5, x2=0.95):
    P = P_kPa*1e3
    h1 = PropsSI("H","P",P,"Q",x1,"Water"); s1 = PropsSI("S","P",P,"Q",x1,"Water")
    h2 = PropsSI("H","P",P,"Q",x2,"Water"); s2 = PropsSI("S","P",P,"Q",x2,"Water")
    m3 = m1+m2
    h3 = (m1*h1 + m2*h2)/m3
    hf = PropsSI("H","P",P,"Q",0,"Water"); hg = PropsSI("H","P",P,"Q",1,"Water")
    if h3 <= hf:
        x3 = 0.0
    elif h3 >= hg:
        x3 = 1.0
    else:
        x3 = PropsSI("Q","P",P,"H",h3,"Water")
    s3 = PropsSI("S","P",P,"Q",x3,"Water") if 0 <= x3 <= 1 else PropsSI("S","P",P,"H",h3,"Water")
    T_sat = PropsSI("T","P",P,"Q",0,"Water") - 273.15

    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    plot_dome_simple(ax)
    puntos = [(s1, f"1: $x_1$={x1:.2f}\nṁ={m1:.1f} kg/s", m1, COL["agua"], (10, 22)),
              (s2, f"2: $x_2$={x2:.2f}\nṁ={m2:.1f} kg/s", m2, COL["agua"], (10, 22)),
              (s3, f"3 (mezcla): $x_3$={x3:.3f}\nṁ={m3:.1f} kg/s", m3, COL["mezcla"], (10, -32))]
    for (sx, nombre, m, color, off) in puntos:
        ax.plot(sx/1e3, T_sat, "o", ms=12*np.sqrt(m/m3) if m3>0 else 8,
                 color=color, mec=COL["tinta"], mew=1.3, zorder=5)
        ax.annotate(nombre, (sx/1e3, T_sat), textcoords="offset points", xytext=off, fontsize=8.3,
                    ha="left" if off[0]>0 else "right")
    ax.plot([s1/1e3, s2/1e3], [T_sat, T_sat], "--", color=COL["tinta"], lw=1, alpha=0.5, zorder=2)
    ax.set_xlabel("$s$ [kJ/(kg·K)]"); ax.set_ylabel("$T$ [°C]")
    ax.set_title(f"Mezcla a $P$={P_kPa:.0f} kPa constante ($T_{{sat}}$={T_sat:.1f}°C)")
    ax.grid(alpha=0.2)
    plt.tight_layout(); plt.show()

def plot_dome_simple(ax, fluid="Water"):
    Tt = PropsSI("Ttriple", fluid)+0.5; Tc_ = PropsSI("Tcrit", fluid)-0.2
    Ts = np.linspace(Tt, Tc_, 200)
    s_l = np.array([PropsSI("S","T",T,"Q",0,fluid) for T in Ts])/1e3
    s_v = np.array([PropsSI("S","T",T,"Q",1,fluid) for T in Ts])/1e3
    ax.plot(s_l, Ts-273.15, color="#777777", lw=1.3); ax.plot(s_v, Ts-273.15, color="#777777", lw=1.3)

interact(camara_mezcla,
         P_kPa=FloatSlider(200.0, min=50.0, max=2000.0, step=50.0, description="P [kPa]"),
         m1=FloatSlider(2.0, min=0.1, max=5.0, step=0.1, description="ṁ₁ [kg/s]"),
         x1=FloatSlider(0.0, min=0.0, max=1.0, step=0.05, description="x₁"),
         m2=FloatSlider(0.5, min=0.1, max=5.0, step=0.1, description="ṁ₂ [kg/s]"),
         x2=FloatSlider(0.95, min=0.0, max=1.0, step=0.05, description="x₂"));

## 7. Instrumentación: cómo se mide esto en una planta real

El modelo completo (§2-§6) parte de 5 cantidades que en una planta real **se miden**, no se calculan: $\dot q_{W,1}$ (caudal de agua), $T_{W,1}$ (temperatura de entrada del agua), $\dot m_{S,1}$ (flujo de vapor, medido típicamente en la turbina), $T_{S,2},p$ (temperatura y presión del vapor) y $T_\infty$ (ambiente) — exactamente las variables de entrada que la §4 (Implementación computacional) ya identificaba como "datos conocidos... la planta térmica cuenta con sensores."

- **Termopares/RTD:** en la entrada y salida de agua ($T_{W,1},T_{W,5}$), y en la carcasa ($T_S$) — si $T_S$ medido se aparta de $T_{sat}(p)$ calculado a partir de la presión medida, es la señal más directa de que hay **gas no condensable** acumulado (aire infiltrado), un problema real y frecuente de operación.
- **Manómetros:** presión de la carcasa (fija $T_{sat}$, ec. del Widget 3) y presión diferencial del agua entre entrada y salida (indica ensuciamiento/obstrucción del haz de tubos).
- **Caudalímetros:** en el agua de enfriamiento (balance de masa, ec. 1) y, si existe una cámara de mezcla (§6), en **cada corriente que se mezcla** — sin caudalímetro en cada rama no hay forma de cerrar el balance de masa $\dot m_3=\dot m_1+\dot m_2$ en operación real.

El widget siguiente marca estos puntos sobre el mismo diagrama del condensador de la §1.

In [ ]:
# WIDGET 6 -- instrumentacion necesaria en el condensador
def dibujar_instrumentos(ax, x, y, tipo, color):
    r = 0.11
    ax.add_patch(Circle((x, y), r, fc="white", ec=color, lw=1.3, zorder=6))
    ax.text(x, y, tipo, ha="center", va="center", fontsize=6.0, zorder=7, fontweight="bold")

def widget_instrumentacion_condensador(mostrar_mezcla=False):
    fig, ax = plt.subplots(figsize=(11, 4.8))
    x0, x1, y0, y1 = 1.2, 9.5, 0.3, 3.2
    ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fc="#f4f1ea", ec=COL["tinta"], lw=1.6))
    n_filas, n_cols = 4, 14
    for yy in np.linspace(y0+0.4, y1-0.4, n_filas):
        for xx in np.linspace(x0+0.5, x1-0.5, n_cols):
            ax.add_patch(Circle((xx, yy), 0.06, fc="white", ec=COL["agua"], lw=0.8, zorder=3))

    ax.add_patch(FancyBboxPatch((x0-0.9, y0-0.1), 0.75, y1-y0+0.2, boxstyle="round,pad=0.03",
                                 fc="#dceefb", ec=COL["agua"], lw=1.4))
    dibujar_instrumentos(ax, x0-1.55, (y0+y1)/2+0.4, "T", COL["perdida"])
    dibujar_instrumentos(ax, x0-1.55, (y0+y1)/2-0.4, "Q", COL["ambiente"])
    ax.text(x0-1.9, (y0+y1)/2-0.9, "agua ent.\n$T_{W,1}$, $\\dot q_{W,1}$", ha="center", fontsize=7.5)

    dibujar_instrumentos(ax, x1+0.55, (y0+y1)/2+0.4, "T", COL["perdida"])
    ax.text(x1+0.55, (y0+y1)/2-0.3, "agua sal.\n$T_{W,5}$", ha="center", fontsize=7.5)

    dibujar_instrumentos(ax, (x0+x1)/2-0.5, y1+0.45, "T", COL["perdida"])
    dibujar_instrumentos(ax, (x0+x1)/2+0.5, y1+0.45, "P", COL["mezcla"])
    ax.text((x0+x1)/2, y1+0.95, "vapor: $T_S$, $p$ (fija $T_{sat}$)", ha="center", fontsize=8, color=COL["tinta"])
    dibujar_instrumentos(ax, (x0+x1)/2-0.9, y1+0.45-0.75, "Q", COL["ambiente"])
    ax.text((x0+x1)/2-0.9, y1+0.45-1.15, "$\\dot m_{S,1}$\n(en turbina)", ha="center", fontsize=7)

    dibujar_instrumentos(ax, (x0+x1)/2, y0-0.85, "T", COL["perdida"])
    ax.text((x0+x1)/2, y0-1.25, "ambiente, $T_\\infty$", ha="center", fontsize=7.5)

    if mostrar_mezcla:
        dibujar_instrumentos(ax, x0+1.0, y1+1.4, "Q", COL["ambiente"])
        dibujar_instrumentos(ax, x0+2.2, y1+1.4, "Q", COL["ambiente"])
        ax.text(x0+1.6, y1+1.75, "caudalímetros en CADA corriente\nde una cámara de mezcla (§6)",
                ha="center", fontsize=7.3, color=COL["ambiente"], style="italic")

    ax.set_xlim(x0-2.2, x1+2.1); ax.set_ylim(y0-1.6, y1+2.1); ax.axis("off"); ax.set_aspect("equal")
    plt.tight_layout(); plt.show()
    print("T = termopar/RTD   P = manómetro   Q = caudalímetro")

interact(widget_instrumentacion_condensador,
         mostrar_mezcla=Checkbox(False, description="incluir cámara de mezcla"));